Suppress all Warnings:

In [ ]:
import warnings
import sys
import logging

# Suppress all warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Suppress logging from libraries
logging.getLogger().setLevel(logging.ERROR)

# Suppress stderr output (optional, extreme)
class DevNull:
    def write(self, msg): pass
    def flush(self): pass

sys.stderr = DevNull()

Seeding:

In [ ]:
import random, numpy as np
import torch

# Set global seeds
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

**1. Loading of Datasets:**

In [ ]:

import pandas as pd

file_path = "reviews.csv"

# Load the dataset with all original columns
df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print(f"Total rows: {len(df)}")
print(f"Columns loaded: {list(df.columns)}")

**2. Schema:**

In [ ]:
# ============================
# SCHEMA SNAPSHOT: df.info()
# ============================

print("Schema Overview:")
df.info()

**3. Data Audit:**

In [ ]:
# ============================
# Overall Data Audit
# ============================

col = 'reply_content'

print(f"\nAudit: '{col}' column")

# Count missing values
missing_count = df[col].isna().sum()
missing_pct = (missing_count / len(df)) * 100

print(f"Missing values: {missing_count} ({missing_pct:.2f}%)")

# Unique non-null values
unique_non_null = df[col].nunique(dropna=True)
print(f"Unique non-null values: {unique_non_null}")

# Most common non-null reply (if any)
if unique_non_null > 0:
    most_common = df[col].dropna().mode()[0]
    most_common_count = df[col].dropna().value_counts().iloc[0]
    print(f"Most common reply: \"{most_common}\" ({most_common_count} occurrences)")
else:
    print("Most common reply: None (all values are NaN)")

# Show a few examples of NaN rows for clarity
print("\nSample rows with NaN reply_content:")
display(df[df[col].isna()].head())

**3a. Audit of Missing Values and Sparse Columns**

In [ ]:
print("Data Audit:")

for col in df.columns:
    print(f"\nColumn: {col}")
    missing_count = df[col].isna().sum()
    missing_pct = (missing_count / len(df)) * 100
    print(f"Missing values: {missing_count} ({missing_pct:.2f}%)")

    if pd.api.types.is_numeric_dtype(df[col]):
        print(" - Numeric Stats:")
        print(f"   * Mean: {df[col].mean():.2f}")
        print(f"   * Median: {df[col].median():.2f}")
        print(f"   * Min: {df[col].min():.2f}")
        print(f"   * Max: {df[col].max():.2f}")
        print(f"   * Std Dev: {df[col].std():.2f}")
    else:
        unique_count = df[col].nunique(dropna=True)
        most_common = df[col].mode()[0] if not df[col].mode().empty else None
        most_common_count = df[col].value_counts().iloc[0] if not df[col].value_counts().empty else 0
        print(f" - Unique values: {unique_count}")
        print(f" - Most common: {most_common} ({most_common_count} occurrences)")

# Validate thumbs_up distribution
zero_thumbs_up_count = (df['thumbs_up'] == 0).sum()
zero_thumbs_up_pct = (zero_thumbs_up_count / len(df)) * 100
print(f"\nPercentage of reviews with 0 thumbs_up: {zero_thumbs_up_pct:.2f}%")

**4. Importing Library and Relevant Attributes Assessment:**

In [ ]:
import pandas as pd

# Load only relevant columns
df = pd.read_csv(
    "reviews.csv",
    usecols=["review_id", "brand", "score", "at", "content",
    "thumbs_up"])

print(df.head())
print(df.info())

**5. Duplicate and Missing Content Audit:**

In [ ]:
# Duplicates across all columns
print("Total duplicate rows:", df.duplicated().sum())

# Missing brand
print ("Missing brand:", df["brand"].isna().sum())

# Missing content
print("Missing content:", df["content"].isna().sum())

# Score validity (should be empty)
invalid_scores = df[~df["score"].between(1, 5)]
print("Invalid scores:\n", invalid_scores)



**6. Data Cleaning & Transformation:**

In [ ]:
# Brand as category
df["brand"] = df["brand"].astype("category")

# Review length
df["review_length"] = df["content"].str.len()
print("Average review length:", df["review_length"].mean())
df["review_length"].describe()



missing timestamps does not affect text modelling and will be retained for completeness

6a. EDA:

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Score distribution
#sns.countplot(x=df["score"])#old
sns.countplot(x="score",data=df)
plt.title("Score Distribution")
plt.show()

# Review length distribution
sns.boxplot(x=df["review_length"])
plt.title("Review Length Distribution")
plt.show()

# Review length by score
sns.boxplot(x=df["score"], y=df["review_length"])
plt.title("Review Length by Score")
plt.show()

# Correlation between score and thumbs_up
corr = df["score"].corr(df["thumbs_up"])
print("Correlation (score vs thumbs_up):", corr)

6b. Exploratory Plot:





In [ ]:
# Defining app-specific technical keywords to check for 'App vs Brand' distinction
app_keywords = ['login', 'crash', 'payment', 'update', 'slow', 'checkout', 'ui', 'bug']

keyword_counts = []
for brand in df['brand'].unique():
    brand_df = df[df['brand'] == brand]
    # Count occurrences of these technical terms per brand
    counts = {kw: brand_df['content'].str.contains(kw, case=False).sum() for kw in app_keywords}
    counts['brand'] = brand
    keyword_counts.append(counts)

keyword_df = pd.DataFrame(keyword_counts).set_index('brand')

# Visualize the distribution of technical issues across the 4 brands
keyword_df.plot(kind='bar', figsize=(12, 7))
plt.title('Pre-Modelling: Frequency of Technical App Issues per Brand')
plt.ylabel('Number of Mentions')
plt.xlabel('Brand')
plt.xticks(rotation=0)
plt.legend(title='Keywords', bbox_to_anchor=(1.05, 1))
plt.show()

6c. Text Pre-processing:

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", "", text)  # remove punctuation & numbers
    tokens = [lemmatizer.lemmatize(w)
        for w in text.split()
        if w not in stop_words]
    return " ".join(tokens)

df["clean_text"] = df["content"].apply(preprocess)

6d. N-Gram Extraction & Brand-wise Comparisons:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_ngrams = CountVectorizer(ngram_range=(2, 3), max_features=50)
X_ngrams = vectorizer_ngrams.fit_transform(df["clean_text"])

ngrams = vectorizer_ngrams.get_feature_names_out()
counts = X_ngrams.sum(axis=0).A1

ngram_freq = sorted(zip(ngrams, counts), key=lambda x: x[1], reverse=True)
print("Top 20 n-grams:")
for ngram, freq in ngram_freq[:20]:
    print(f"{ngram}: {freq}")

6e. Post-Processing EDA:Bigrams Analysis:




In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_top_bigrams(corpus, top_k=15):
    # n-gram_range=(2, 2) focuses on word pairs to preserve semantic meaning
    vec = CountVectorizer(ngram_range=(2, 2)).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)

    top_df = pd.DataFrame(words_freq[:top_k], columns=['Phrase', 'Frequency'])

    plt.figure(figsize=(10, 8))
    sns.barplot(x='Frequency', y='Phrase', data=top_df, palette='viridis')
    plt.title('Top 15 Bigrams: Evidence of App-Specific Feedback')
    plt.show()

plot_top_bigrams(df['clean_text'])

Stopwords, punctuation removed and Lemmatisation applied
Latter is chosen over stemmer to preserve the semantics of
the words/phrases.

**7. Prepare Data for Topic Modelling:**

7a. Installing gensim:

In [ ]:
!pip install gensim

7b. Vectorisation (BOW):

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Vectorise cleaned text
vectorizer = CountVectorizer(
    max_df=0.95,
    min_df=5,
    stop_words='english')

bow_matrix = vectorizer.fit_transform(df["clean_text"])
id2word = vectorizer.get_feature_names_out()

7c. Prepare corpus for Gensim LDA:

In [ ]:
from gensim.corpora import Dictionary

texts = [t.split() for t in df["clean_text"]]
dictionary = Dictionary(texts)
corpus = [dictionary.doc2bow(t) for t in texts]

**8. LDA Topic Modelling:**

8a. LDA Hyperparameter Tuning:

In [ ]:
from gensim.models.ldamodel import LdaModel

topic_nums = [5, 10, 15]
lda_models = {}

for k in topic_nums:
    lda_models[k] = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        passes=20,
        random_state=42)

8b. Coherence Evaluation:

In [ ]:
from gensim.models import CoherenceModel

coherence_scores = {}

for k, model in lda_models.items():
    cm = CoherenceModel(
        model=model,
        texts=texts,
        dictionary=dictionary,
        coherence='c_v')
    coherence_scores[k] = cm.get_coherence()
    print(f"Coherence for {k} topics: {coherence_scores[k]:.4f}")

In [ ]:
print("\nExtracted Topics for 5 vs 10:")

print("\n--- 5 Topics ---")
for idx, topic in lda_models[5].show_topics(num_topics=5, num_words=10, formatted=False):
    words = [w for w, p in topic]
    print(f"Topic {idx}: {', '.join(words)}")

print("\n--- 10 Topics ---")
for idx, topic in lda_models[10].show_topics(num_topics=10, num_words=10, formatted=False):
    words = [w for w, p in topic]
    print(f"Topic {idx}: {', '.join(words)}")

8c. Perplexity Evaluation:

In [ ]:
perplexity_scores = {}

for k, model in lda_models.items():
    perplexity_scores[k] = model.log_perplexity(corpus)
    print(f"Perplexity for {k} topics: {perplexity_scores[k]}")

8d. Select best Topic Number (based on Coherence):

In [ ]:
best_k = max(coherence_scores, key=coherence_scores.get)
best_lda = lda_models[best_k]

print(f"Selected number of topics: {best_k}")

8e. Display Top Words for Each Topic:

In [ ]:
# Collect topics into a structured list
topic_rows = []
for idx, topic in best_lda.show_topics(num_topics=best_k, num_words=10, formatted=False):
    words = [w for w, p in topic]
    topic_rows.append({
        "Topic ID": idx,
        "Top Words": ", ".join(words)
    })

# Convert to DataFrame for neat display
topics_df = pd.DataFrame(topic_rows)
print(topics_df)

8f. pyLDAvis Visualisation:

In [ ]:
!pip install pyLDAvis

import pyLDAvis
import pyLDAvis.gensim

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(best_lda, corpus, dictionary)
vis

8g. Assign Dominant Topic to Each Review:

In [ ]:
doc_topics = [best_lda.get_document_topics(bow) for bow in corpus]

dominant_topics = []
for doc in doc_topics:
    if doc:
        dominant_topics.append(max(doc, key=lambda x: x[1])[0])
    else:
        dominant_topics.append(None)

df["dominant_topic"] = dominant_topics

8h. Brand-Wise Topic Distribution:

In [ ]:
brand_topic_counts = df.groupby(["brand", "dominant_topic"]).size().unstack(fill_value=0)
brand_topic_counts


In [ ]:
brand_topic_counts.plot(kind='bar', stacked=True, figsize=(10, 6))
plt.title("Distribution of Topics Across Sportswear Brands")
plt.ylabel("Number of Reviews")
plt.legend(title="Topic ID", bbox_to_anchor=(1.05, 1))
plt.show()

In [ ]:
#saving LDA modelling

# Saving each LDA model in the dictionary
for k, model in lda_models.items():
    model.save(f"lda_model_{k}_topics")

**9. BERTopic Modelling:**

9a. Import and install BERTopic Components:

In [ ]:
!pip install bertopic sentence-transformers umap-learn hdbscan

In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# --- Imports ---
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from sentence_transformers import SentenceTransformer

# --- Define embedding model ---
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")  # lightweight, fast

# --- Initialize results dictionary ---
brand_results = {}

# --- Process each brand ---
for b in ["nike", "adidas", "puma", "gymshark"]:
    df_b = df[df["brand"] == b]

    # LDA
    texts_b = [t.split() for t in df_b["clean_text"]]
    dictionary_b = Dictionary(texts_b)
    corpus_b = [dictionary_b.doc2bow(t) for t in texts_b]
    lda_b = LdaModel(corpus=corpus_b, id2word=dictionary_b,
                     num_topics=5, passes=20, random_state=42)
    lda_topics_b = lda_b.show_topics(num_topics=5, num_words=10, formatted=False)

    # BERTopic (using defaults for ctfidf, umap, hdbscan unless you tuned them earlier)
    docs_b = df_b["clean_text"].tolist()
    vectorizer_model_b = CountVectorizer(stop_words="english", min_df=1, max_df=0.95)

    topic_model_b = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer_model_b,
        verbose=True)

    topics_b, probs_b = topic_model_b.fit_transform(docs_b)
    topic_info_b = topic_model_b.get_topic_info()

    brand_results[b] = {
        "lda_model": lda_b,
        "lda_topics": lda_topics_b,
        "bert_model": topic_model_b,
        "bert_topics": topic_info_b,
        "bert_topic_assignments": topics_b }

print("Brands processed:", brand_results.keys())

In [ ]:
from bertopic.vectorizers import ClassTfidfTransformer

ctfidf_model = ClassTfidfTransformer()

topic_model_b = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model_b,
    ctfidf_model=ctfidf_model,
    verbose=True)

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from sentence_transformers import SentenceTransformer

# Define embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

brand_results = {}

for b in ["nike", "adidas", "puma", "gymshark"]:
    df_b = df[df["brand"] == b]

    # LDA
    texts_b = [t.split() for t in df_b["clean_text"]]
    dictionary_b = Dictionary(texts_b)
    corpus_b = [dictionary_b.doc2bow(t) for t in texts_b]
    lda_b = LdaModel(corpus=corpus_b, id2word=dictionary_b,
                     num_topics=5, passes=20, random_state=42)
    lda_topics_b = lda_b.show_topics(num_topics=5, num_words=10, formatted=False)

    # BERTopic (defaults for ctfidf, umap, hdbscan)
    docs_b = df_b["clean_text"].tolist()
    vectorizer_model_b = CountVectorizer(stop_words="english", min_df=1, max_df=0.95)

    topic_model_b = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer_model_b,
        verbose=True)

    topics_b, probs_b = topic_model_b.fit_transform(docs_b)
    topic_info_b = topic_model_b.get_topic_info()

    brand_results[b] = {
        "lda_model": lda_b,
        "lda_topics": lda_topics_b,
        "bert_model": topic_model_b,
        "bert_topics": topic_info_b,
        "bert_topic_assignments": topics_b}

print("Brands processed:", brand_results.keys())


In [ ]:
# Standardize BERTopic output headers
brand_results[b]["bert_topics"].rename(columns={
    "Topic": "Topic ID",
    "Name": "Topic Label",
    "Representation": "Top Keywords",
    "Count": "Document Count"}, inplace=True)

In [ ]:
from bertopic import BERTopic
topic_model = BERTopic()

from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer

9b. BERTopic Topic Modelling - Base Run:

In [ ]:
docs = df["clean_text"].tolist()

In [ ]:
# Rebuild Base Model (Round 0)
embedding_model_base = SentenceTransformer("all-MiniLM-L6-v2")

topic_model_base = BERTopic(
    embedding_model=embedding_model_base,
    verbose=True)

topics_base, probs_base = topic_model_base.fit_transform(docs)
topic_info_base = topic_model_base.get_topic_info()
topic_info_base

In [ ]:
topic_model_base.save("bertopic_base_model")

9c. Build Custom BERTopic Pipeline:

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

umap_model = UMAP(
    n_neighbors=15,


    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42)

hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    min_samples=15,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True)

vectorizer_model = CountVectorizer(
    ngram_range=(1, 3),
    stop_words="english")

ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True)

topic_model_base = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    verbose=True)

9d. Fit BERTopic:

In [ ]:
topics_base, probs_base = topic_model_base.fit_transform(docs)

9e. Topic Summary Table:

In [ ]:
topic_info = topic_model_base.get_topic_info()
topic_info

9f. Outlier Analysis (Base Run) :

In [ ]:
topic_info[topic_info["Topic"] == -1]

9g. BERTopic Topic Modelling - Tuned Run:

In [ ]:
# Tuning Round 1 — UMAP + HDBSCAN
umap_model_tuned = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42)

hdbscan_model_tuned = HDBSCAN(
    min_cluster_size=30,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True)

# Build tuned BERTopic model
topic_model_tuned = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model_tuned,
    hdbscan_model=hdbscan_model_tuned,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    verbose=True)

# Fit tuned model
topics_tuned, probs_tuned = topic_model_tuned.fit_transform(docs)

# Get tuned topic info
topic_info_tuned = topic_model_tuned.get_topic_info()
topic_info_tuned

In [ ]:
topic_model_round1 = topic_model_tuned
topic_model_round1.save("bertopic_round1_model")
topic_info_round1 = topic_model_round1.get_topic_info()

9h. Outlier Analysis:

In [ ]:
topic_info_tuned[topic_info_tuned["Topic"] == -1]

In [ ]:
#Saving Round 1
topic_model_round1.save("bertopic_round1_model")



9i Tuning Round 2:

In [ ]:
topic_info_tuned.sort_values("Count", ascending=False).head(10)

In [ ]:
# Use MiniLM again — faster, more stable
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Restore stable vectorizer
vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    min_df=20,
    stop_words="english")

# Rebuild model with tuned UMAP + HDBSCAN
topic_model_tuned = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model_tuned,
    hdbscan_model=hdbscan_model_tuned,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    verbose=True)

# Refit model
topics_tuned, probs_tuned = topic_model_tuned.fit_transform(docs)

# Get topic info
topic_info_tuned = topic_model_tuned.get_topic_info()
topic_info_tuned

In [ ]:
topic_model_round2 = topic_model_tuned

In [ ]:
#Saving Round 2 Tuning
topic_model_round2.save("bertopic_round2_model")

In [ ]:
topic_info_tuned[topic_info_tuned["Topic"] == -1]

9j. Intertopic Distance Map:

In [ ]:
topic_model_round1.visualize_topics()

9k. Hierarchical Topic Tree:

In [ ]:
topic_model_round1.visualize_hierarchy()

9l. Topic Size Distribution:

In [ ]:
topic_model_round1.visualize_barchart()

9m. Representative Documents

In [ ]:
topic_model_round1.get_representative_docs()

9n. Brand-Level Modelling:

In [ ]:
# BRAND-LEVEL MODELLING
from gensim.corpora import Dictionary
from gensim.models.ldamodel import LdaModel

brands = df["brand"].unique()
brand_results = {}

for b in brands:
    print(f"\nProcessing brand: {b}")
    df_b = df[df["brand"] == b]

    # ----- LDA for this brand -----
    texts_b = [t.split() for t in df_b["clean_text"]]
    dictionary_b = Dictionary(texts_b)
    corpus_b = [dictionary_b.doc2bow(t) for t in texts_b]

    lda_b = LdaModel(
        corpus=corpus_b,
        id2word=dictionary_b,
        num_topics=5,
        passes=20,
        random_state=42)

    lda_topics_b = lda_b.show_topics(num_topics=5, num_words=10, formatted=False)

    # ----- BERTopic for this brand -----
    docs_b = df_b["clean_text"].tolist()
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model_b = CountVectorizer(
    stop_words="english",
    min_df=1,
    max_df=0.95)

topic_model_b = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model_b,
    ctfidf_model=ctfidf_model,
    umap_model=umap_model_tuned,
    hdbscan_model=hdbscan_model_tuned,
    verbose=True)

docs_b = df_b["clean_text"].tolist()


topics_b, probs_b = topic_model_b.fit_transform(docs_b)
topic_info_b = topic_model_b.get_topic_info()

brand_results[b] = {
    "lda_model": lda_b,
    "lda_topics": lda_topics_b,
    "bert_model": topic_model_b,
    "bert_topics": topic_info_b,
    "bert_topic_assignments": topics_b}


9o. Top 3 Topics per Brand:

In [ ]:
rows = []
for b in brand_results.keys():
    # Get top 3 topics for each brand
    topic_info = brand_results[b]["bert_topics"].head(3)

    for _, row in topic_info.iterrows():
        rows.append({
            "Brand": b,
            "Topic ID": row.get("Topic ID", row.get("Topic")),
            "Topic Label": row.get("Topic Label", row.get("Name")),
            "Top Keywords": ", ".join(row.get("Top Keywords", row.get("Representation"))[:5])
                           if isinstance(row.get("Top Keywords", row.get("Representation")), list)
                           else row.get("Top Keywords", row.get("Representation")),
            "Document Count": row.get("Document Count", row.get("Count"))
        })

comparison_df = pd.DataFrame(rows)
comparison_df

In [ ]:
#Saving BERTopic Modelling

topic_model_tuned.save("bertopic_final_model")

**Nike Brand's Theme:**

In [ ]:
b = "nike"
df_b = df[df["brand"] == b]

texts_b = [t.split() for t in df_b["clean_text"]]
dictionary_b = Dictionary(texts_b)
corpus_b = [dictionary_b.doc2bow(t) for t in texts_b]

lda_b = LdaModel(
    corpus=corpus_b,
    id2word=dictionary_b,
    num_topics=5,
    passes=20,
    random_state=42)

lda_topics_b = lda_b.show_topics(num_topics=5, num_words=10, formatted=False)

vectorizer_model_b = CountVectorizer(
    stop_words="english",
    min_df=1,
    max_df=0.95)

topic_model_b = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model_b,
    ctfidf_model=ctfidf_model,
    umap_model=umap_model_tuned,
    hdbscan_model=hdbscan_model_tuned,
    verbose=True)

docs_b = df_b["clean_text"].tolist()
topics_b, probs_b = topic_model_b.fit_transform(docs_b)
topic_info_b = topic_model_b.get_topic_info()

brand_results[b] = {
    "lda_model": lda_b,
    "lda_topics": lda_topics_b,
    "bert_model": topic_model_b,
    "bert_topics": topic_info_b,
    "bert_topic_assignments": topics_b}

In [ ]:
brand_results["nike"]["bert_topics"].head(10)

**Adidas Brand's Theme:**

In [ ]:
b = "adidas"
df_b = df[df["brand"] == b]

texts_b = [t.split() for t in df_b["clean_text"]]
dictionary_b = Dictionary(texts_b)
corpus_b = [dictionary_b.doc2bow(t) for t in texts_b]

lda_b = LdaModel(
    corpus=corpus_b,
    id2word=dictionary_b,
    num_topics=5,
    passes=20,
    random_state=42)

lda_topics_b = lda_b.show_topics(num_topics=5, num_words=10, formatted=False)

vectorizer_model_b = CountVectorizer(
    stop_words="english",
    min_df=1,
    max_df=0.95)

topic_model_b = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model_b,
    ctfidf_model=ctfidf_model,
    umap_model=umap_model_tuned,
    hdbscan_model=hdbscan_model_tuned,
    verbose=True)

docs_b = df_b["clean_text"].tolist()
topics_b, probs_b = topic_model_b.fit_transform(docs_b)
topic_info_b = topic_model_b.get_topic_info()

brand_results[b] = {
    "lda_model": lda_b,
    "lda_topics": lda_topics_b,
    "bert_model": topic_model_b,
    "bert_topics": topic_info_b}

In [ ]:
brand_results["adidas"]["bert_topics"].head(10)

**Puma's Brand's Theme:**

In [ ]:
b = "puma"
df_b = df[df["brand"] == b]

texts_b = [t.split() for t in df_b["clean_text"]]
dictionary_b = Dictionary(texts_b)
corpus_b = [dictionary_b.doc2bow(t) for t in texts_b]

lda_b = LdaModel(
    corpus=corpus_b,
    id2word=dictionary_b,
    num_topics=5,
    passes=20,
    random_state=42)

lda_topics_b = lda_b.show_topics(num_topics=5, num_words=10, formatted=False)

vectorizer_model_b = CountVectorizer(
    stop_words="english",
    min_df=1,
    max_df=0.95)

topic_model_b = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model_b,
    ctfidf_model=ctfidf_model,
    umap_model=umap_model_tuned,
    hdbscan_model=hdbscan_model_tuned,
    verbose=True)

docs_b = df_b["clean_text"].tolist()
topics_b, probs_b = topic_model_b.fit_transform(docs_b)
topic_info_b = topic_model_b.get_topic_info()

brand_results[b] = {
    "lda_model": lda_b,
    "lda_topics": lda_topics_b,
    "bert_model": topic_model_b,
    "bert_topics": topic_info_b}

In [ ]:
brand_results["puma"]["bert_topics"].head(20)

**Gymshark Brand's Theme:**

In [ ]:
b = "gymshark"
df_b = df[df["brand"] == b]

texts_b = [t.split() for t in df_b["clean_text"]]
dictionary_b = Dictionary(texts_b)
corpus_b = [dictionary_b.doc2bow(t) for t in texts_b]

lda_b = LdaModel(
    corpus=corpus_b,
    id2word=dictionary_b,
    num_topics=5,
    passes=20,
    random_state=42)

lda_topics_b = lda_b.show_topics(num_topics=5, num_words=10, formatted=False)

vectorizer_model_b = CountVectorizer(
    stop_words="english",
    min_df=1,
    max_df=0.95)

topic_model_b = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model_b,
    ctfidf_model=ctfidf_model,
    umap_model=umap_model_tuned,
    hdbscan_model=hdbscan_model_tuned,
    verbose=True)

docs_b = df_b["clean_text"].tolist()
topics_b, probs_b = topic_model_b.fit_transform(docs_b)
topic_info_b = topic_model_b.get_topic_info()

brand_results[b] = {
    "lda_model": lda_b,
    "lda_topics": lda_topics_b,
    "bert_model": topic_model_b,
    "bert_topics": topic_info_b}

In [ ]:
brand_results["gymshark"]["bert_topics"].head(10)

**10. LDA and BERT Comparisons**

In [ ]:
def compare_lda_bert(b):
    lda_topics = brand_results[b]["lda_topics"]
    bert_topics = brand_results[b]["bert_topics"].head(5)

    rows = []

    # LDA topics
    for topic_id, words in lda_topics:
        rows.append({
            "Brand": b,
            "Model": "LDA",
            "Topic ID": topic_id,
            "Keywords": ", ".join([w for w, _ in words])})

    # BERT topics
    for _, row in bert_topics.iterrows():
        rows.append({
            "Brand": b,
            "Model": "BERT",
            "Topic ID": row["Topic"],
            "Keywords": ", ".join(row["Representation"][:10])})

    return pd.DataFrame(rows)

In [ ]:
all_rows = []

for b in brand_results.keys():
    df_compare = compare_lda_bert(b)
    all_rows.append(df_compare)

full_compare_df = pd.concat(all_rows, ignore_index=True)
full_compare_df

10a. Topic Labelling:

In [ ]:
topic_labels = {
    0: "Item Availability, Delivery & Product Issues",
    1: "App Experience & Product Satisfaction",
    2: "Customer Service & Order Issues",
    3: "Product Satisfaction & Fitness Apparel",
    4: "Checkout / Payment / Price Issues"}

In [ ]:
df["topic_label"] = df["dominant_topic"].map(topic_labels)

In [ ]:
df["topic_label"].value_counts(normalize=True)

In [ ]:
print(type(cleaned_texts))
print(len(cleaned_texts))
print(cleaned_texts[:2])

10b. Extact Top 5 BERTopics:

In [ ]:
from bertopic import BERTopic

# Fit BERTopic on your cleaned text corpus

topic_model = BERTopic()
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(cleaned_texts)

# Get top 5 topics
topic_info = topic_model.get_topic_info()
print(topic_info.head(5))

# Inspect one topic
print(topic_model.get_topic(0))

In [ ]:
import pandas as pd

# Load your dataset (adjust path if needed)
df = pd.read_csv(
    "reviews.csv",
    usecols=["review_id", "brand", "score", "at", "content", "thumbs_up"]
)

# Quick check
print(df.head())

11a. Read Relevant Columns:

In [ ]:
df = pd.read_csv(
    "reviews.csv",
    usecols=["review_id", "brand", "score", "at", "content", "thumbs_up"])


11b. Including 'reply_content':

In [ ]:
replies = pd.read_csv(
    "reviews.csv",
    usecols=["review_id", "brand", "reply_content"])

11c. Adding Reply Length:

In [ ]:
replies["reply_length"] = replies["reply_content"].astype(str).str.len()

11d. Merging 'reply_content':

In [ ]:
df = df.merge(
    replies[["review_id", "reply_content"]],
    on="review_id",
    how="left")

In [ ]:
df['reply_content'].isna().value_counts()
df.groupby('brand')['reply_content'].apply(lambda x: x.notna().sum())

11e. Reviewing "Reply_content" Overview

In [ ]:
df['reply_content'].isna().value_counts()
df.groupby('brand')['reply_content'].apply(lambda x: x.notna().sum())

In [ ]:
print(df.columns)

11f. Filtering for adidas and Gymshark only:

In [ ]:
# Filter for adidas and Gymshark only
replies_filtered = df[df["brand"].isin(["adidas", "gymshark"])]

# Drop rows with missing replies
replies_filtered = replies_filtered.dropna(subset=["reply_content"])

# Add reply_length column
replies_filtered = replies_filtered.assign(
    reply_length=replies_filtered["reply_content"].str.len()
)

# Truncate reply_content for neat display
replies_filtered["reply_preview"] = replies_filtered["reply_content"].str.slice(0, 80) + "..."

# Preview the first 10 rows
print(replies_filtered[["brand", "reply_preview", "reply_length"]].head(10))

**12. Tracking emojis:**

In [ ]:
# emojis in 'reply_content'
import re

# Function to detect emojis
def extract_emojis(text):
    if isinstance(text, str):
        return re.findall(r'[\U00010000-\U0010ffff]', text)
    return []

# Apply to reply_content
df["emojis"] = df["reply_content"].apply(extract_emojis)

# Count emojis per brand
emoji_stats = (
    df.groupby("brand")["emojis"]
      .apply(lambda x: sum(len(e) for e in x))
      .reset_index(name="emoji_count")
)

# Also calculate % of replies with at least one emoji
emoji_stats["emoji_reply_ratio"] = (
    df.groupby("brand")["emojis"]
      .apply(lambda x: sum(1 for e in x if len(e) > 0) / len(x))
      .values
)

print(emoji_stats)

concentrated in Gymshark only

In [ ]:
#emojis in 'content'
import re

def contains_emoji(text):
    if isinstance(text, str):
        return bool(re.search(r'[\U00010000-\U0010ffff]', text))
    return False

# Flag reviews with emojis
df["content_has_emoji"] = df["content"].apply(contains_emoji)

# Count per brand
emoji_in_content = df.groupby("brand")["content_has_emoji"].mean().reset_index()
emoji_in_content["content_has_emoji"] = emoji_in_content["content_has_emoji"] * 100

print(emoji_in_content)

In [ ]:
Numbers immaterial most <= 10% of 6,446 reviews

Overall Emojis in %:

In [ ]:
overall_ratio = df["content_has_emoji"].mean() * 100
print(overall_ratio)